## Задание 1


1) Использовать датасет MovieLens.
2) Построить рекомендации (регрессия, предсказываем оценку) на фичах:
- TF-IDF на тегах и жанрах;
- средние оценки (+ median, variance и т. д.) пользователя и фильма.
3) Оценить RMSE на тестовой выборке.

In [30]:
import pandas as pd
import numpy as np
from tqdm import tqdm_notebook
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

Выгрузка данных

In [2]:
data_movies = pd.read_csv('/Users/sofagusina/Desktop/программирование/machine_learning/machine_learning/ML/RS/content-based/ml-latest-small/movies.csv')
data_ratings = pd.read_csv('/Users/sofagusina/Desktop/программирование/machine_learning/machine_learning/ML/RS/content-based/ml-latest-small/ratings.csv')
data_tags = pd.read_csv('/Users/sofagusina/Desktop/программирование/machine_learning/machine_learning/ML/RS/content-based/ml-latest-small/tags.csv')
data_links = pd.read_csv('/Users/sofagusina/Desktop/программирование/machine_learning/machine_learning/ML/RS/content-based/ml-latest-small/links.csv')

In [3]:
data_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
data_ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
data_tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [6]:
data_movies_with_ratings = data_movies.join(data_ratings.set_index('movieId'), on='movieId').reset_index(drop=True)

In [7]:
tags_grouped = data_tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.astype(str))).reset_index()

In [8]:
data_movies_with_ratings_tags = data_movies_with_ratings.merge(
    data_tags, on="movieId", how="left", suffixes=("", "_tag")
)
data_movies_with_ratings_tags = data_movies_with_ratings_tags.dropna()
columns_drop = ['userId_tag','timestamp_tag']
data_movies_with_ratings_tags = data_movies_with_ratings_tags.drop(columns=columns_drop)

In [9]:
data_movies_with_ratings_tags

,movieId,title,genres,userId,rating,timestamp,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,4.0,9.649827e+08,pixar
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,4.0,9.649827e+08,pixar
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,4.0,9.649827e+08,fun
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,4.0,8.474350e+08,pixar
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,4.0,8.474350e+08,pixar
...,...,...,...,...,...,...,...
285745,187595,Solo: A Star Wars Story (2018),Action|Adventure|Children|Sci-Fi,586.0,5.0,1.529900e+09,star wars
285770,193565,Gintama: The Movie (2010),Action|Animation|Comedy|Sci-Fi,184.0,3.5,1.537099e+09,anime
285771,193565,Gintama: The Movie (2010),Action|Animation|Comedy|Sci-Fi,184.0,3.5,1.537099e+09,comedy
285772,193565,Gintama: The Movie (2010),Action|Animation|Comedy|Sci-Fi,184.0,3.5,1.537099e+09,gintama


In [10]:
def change_string(s):
    return ' '.join(s.replace(' ', '').replace('-', '').split('|'))

Подготовка фич

1. Подготовка TD-IDF на тегах и жанрах
- Берём жанры + теги фильма,
- Считаем TF-IDF,
- Получаем вектор признаков фильма (например, длиной 5000), где каждое число отражает значимость конкретного слова (жанра/тега) для этого фильма.

In [11]:
tag_strings = []
movies = []
genres = []

for movie, group in tqdm_notebook(data_movies_with_ratings_tags.groupby('title')):
    tags = group.tag.dropna()
    tag_str = ' '.join([str(s).replace('-', '') for s in tags]) if not tags.empty else ''
    tag_strings.append(tag_str)
    
    movies.append(movie)

    genre_str = group.genres.values[0] if pd.notna(group.genres.values[0]) else ''
    genres.append(change_string(genre_str))

/var/folders/07/xx2bf7ss4g30z8mzlm1lcbqr0000gn/T/ipykernel_10524/1884474423.py:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for movie, group in tqdm_notebook(data_movies_with_ratings_tags.groupby('title')):


  0%|          | 0/1554 [00:00<?, ?it/s]

In [12]:
movies_tags_genres_filtered = pd.DataFrame(
    {
        "movie": movies,
        "tag": tag_strings,
        'genres': genres
    }
)

In [13]:
movies_tags_genres_filtered["movie_text"] = movies_tags_genres_filtered["genres"] + ' ' + movies_tags_genres_filtered["tag"]

In [14]:
movies_tags_genres_filtered["movieId"] = data_movies_with_ratings_tags.groupby("title")["movieId"].first().values

In [42]:
movies_tags_genres_filtered

,movie,tag,genres,movie_text,movieId
0,(500) Days of Summer (2009),artistic Funny humorous inspiring intelligent ...,Comedy Drama Romance,Comedy Drama Romance artistic Funny humorous i...,69757
1,...And Justice for All (1979),lawyers lawyers lawyers,Drama Thriller,Drama Thriller lawyers lawyers lawyers,3420
2,10 Cloverfield Lane (2016),creepy suspense creepy suspense creepy suspens...,Thriller,Thriller creepy suspense creepy suspense creep...,152077
3,10 Things I Hate About You (1999),Shakespeare sort of Shakespeare sort of Shakes...,Comedy Romance,Comedy Romance Shakespeare sort of Shakespeare...,2572
4,101 Dalmatians (1996),dogs remake dogs remake dogs remake dogs remak...,Adventure Children Comedy,Adventure Children Comedy dogs remake dogs rem...,1367
...,...,...,...,...,...
1549,Zero Dark Thirty (2012),Afghanistan American propaganda assassination ...,Action Drama Thriller,Action Drama Thriller Afghanistan American pro...,98961
1550,Zombieland (2009),Bill Murray dark comedy Emma Stone funny Jesse...,Action Comedy Horror,Action Comedy Horror Bill Murray dark comedy E...,71535
1551,Zoolander (2001),ben stiller comedy David Bowie goofy mindless ...,Comedy,Comedy ben stiller comedy David Bowie goofy mi...,4816
1552,Zulu (1964),Africa Africa Africa Africa,Action Drama War,Action Drama War Africa Africa Africa Africa,5899


In [15]:
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(movies_tags_genres_filtered['movie_text'])
X_train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf.get_feature_names_out())
X_train_tfidf_df["movieId"] = movies_tags_genres_filtered["movieId"].values

2. Усредненные оценки пользователя и фильма

In [16]:
user_stats = data_movies_with_ratings_tags.groupby("userId")["rating"].agg(["mean","median","var","count"]).reset_index()
user_stats.columns = ["userId","user_mean","user_median","user_var","user_count"]

In [17]:
user_stats

,userId,user_mean,user_median,user_var,user_count
0,1.0,4.040472,4.0,1.086197,593
1,2.0,4.020979,4.0,0.379838,143
2,3.0,0.500000,0.5,0.000000,27
3,4.0,2.708621,2.0,2.472810,580
4,5.0,4.517986,5.0,0.654910,278
...,...,...,...,...,...
605,606.0,4.109808,4.0,0.561369,1407
606,607.0,3.533333,3.0,0.783964,450
607,608.0,3.805837,4.0,0.964802,1285
608,609.0,3.817427,4.0,0.149862,241


In [18]:
movie_stats = data_movies_with_ratings_tags.groupby("movieId")["rating"].agg(["mean","median","var","count"]).reset_index()
movie_stats.columns = ["movieId","movie_mean","movie_median","movie_var","movie_count"]

In [19]:
movie_stats

,movieId,movie_mean,movie_median,movie_var,movie_count
0,1,3.920930,4.0,0.694825,645
1,2,3.431818,3.5,0.772106,440
2,3,3.259615,3.0,1.101848,104
3,5,3.071429,3.0,0.814433,98
4,7,3.185185,3.0,0.955625,54
...,...,...,...,...,...
1549,183611,4.000000,4.0,0.000000,3
1550,184471,2.500000,3.0,1.500000,12
1551,187593,3.875000,4.0,1.419643,36
1552,187595,3.900000,4.0,0.488889,10


In [20]:
for df_ in [data_movies_with_ratings_tags, user_stats, movie_stats]:
    if 'userId' in df_.columns:
        df_['userId'] = df_['userId'].astype(int)
    if 'movieId' in df_.columns:
        df_['movieId'] = df_['movieId'].astype(int)

In [21]:
df_clean = data_movies_with_ratings_tags.dropna(subset=["userId","movieId","rating"])

df_clean = df_clean.merge(user_stats, on="userId", how="left")
df_clean = df_clean.merge(movie_stats, on="movieId", how="left")

In [23]:
num_cols = ["user_mean","user_median","user_var","user_count",
            "movie_mean","movie_median","movie_var","movie_count"]
df_clean[num_cols] = df_clean[num_cols].fillna(0)

In [25]:
numeric_features_sparse = csr_matrix(df_clean[num_cols].values)

3. Склеиваем две фичи

In [26]:
movieid_to_index = dict(zip(movies_tags_genres_filtered["movieId"], range(len(movies_tags_genres_filtered))))

# Индексы TF-IDF для каждой строки df_clean
tfidf_indices = df_clean["movieId"].map(movieid_to_index).values
X_tfidf_selected = X_train_tfidf[tfidf_indices]

# Объединяем sparse матрицы
X = hstack([X_tfidf_selected, numeric_features_sparse])

# Целевая переменная
y = df_clean["rating"].values

Обучение модели линейной регрессии

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


# RMSE
rmse = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE:", rmse)

RMSE: 0.7722572283138666


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


4. Предскажем рейтинг фильма по данным тегам 'pixar|pixar|fun'

In [40]:
test_tags = change_string('pixar|pixar|fun')

test_tfidf = tfidf.transform([test_tags])

## возьмем пользователя со средними характеристиками
user_features = user_stats[["user_mean","user_median","user_var","user_count"]].mean().to_numpy().reshape(1, -1)
movie_features = movie_stats[["movie_mean","movie_median","movie_var","movie_count"]].mean().to_numpy().reshape(1, -1)

user_features_sparse = csr_matrix(user_features)
movie_features_sparse = csr_matrix(movie_features)

X_test_input = hstack([test_tfidf, user_features_sparse, movie_features_sparse])

y_pred_test = model.predict(X_test_input)
print("Predicted rating:", y_pred_test[0])

Predicted rating: 3.6535327412179552
